# 00 — Understanding Praval Architecture

**Mode:** Offline  
**Level:** Fundamentals  
**Estimated time:** 20 minutes

[Watch the original lesson video](https://www.youtube.com/watch?v=M30U-6w_WGc)

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

A tiny message network: one producer publishes a structured Spore, the Reef routes it, and a handler records what arrived. You will inspect the message, the execution timeline, and the Reef's network statistics.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(0, 'Understanding Praval Architecture')


## Prerequisites and setup

**Course prerequisites:** None. This is the starting point.

**Execution requirements:** Offline. No API key or external service is required.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Explain the roles of Agent, Reef, Spore, handler, and model runtime.
- Send a real Spore through an isolated Reef.
- Wait for handler completion and inspect the resulting runtime state.
- Shut down the Reef cleanly.


## Mental model

An **Agent** owns a specialized responsibility. A **handler** is the code it runs when an interesting message arrives. A **Spore** is the structured message. The **Reef** routes Spores to subscribers. A **model runtime** is optional: it is used only when an agent needs a language model. The routing system itself works without one.


In [ ]:
show_flow(
('Producer', 'creates knowledge', 'agent'),
('Spore', 'carries fields', 'spore'),
('Reef', 'routes by channel', 'reef'),
('Handler', 'does focused work', 'agent'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Create an isolated Reef

A fresh Reef keeps this lesson independent from any global agent network. The list will hold the exact Spore object observed by the handler.


In [ ]:
from praval.core.reef import Reef

reef = Reef()
received = []
show_json(reef.get_network_stats(), "Empty Reef")


### What just happened?

`Reef()` created in-memory channel infrastructure, but no subscribers or messages exist yet.

### Why this matters

Isolating a workflow makes lifecycle and test behavior predictable.


### Register a handler

A subscription connects an agent name and a Python callable to a channel. The callable receives one Spore at a time.


In [ ]:
def architecture_observer(spore):
    stage("observer", "handler started", spore.knowledge["type"], spore)
    received.append(spore)
    stage("observer", "handler finished", "knowledge captured", spore)


reef.subscribe(
    "architecture-observer",
    architecture_observer,
    channel=reef.default_channel,
)
show_json(reef.get_network_stats(), "Reef after subscription")


### What just happened?

The Reef now knows which handler to schedule on its default channel. The function has not run yet.

### Why this matters

Subscriptions separate message producers from consumers. Neither side needs a direct reference to the other.


### Broadcast one Spore

Broadcasting turns the knowledge dictionary into a real Spore and schedules the matching handler.


In [ ]:
stage("learner", "broadcast requested", "architecture_question")
spore_id = reef.broadcast(
    from_agent="learner",
    knowledge={
        "type": "architecture_question",
        "question": "How do agents exchange knowledge?",
    },
)
stage("reef", "Spore accepted", spore_id)

completed = reef.wait_for_completion(timeout=10)
assert completed and len(received) == 1


### What just happened?

`broadcast()` returned the Spore ID immediately. `wait_for_completion()` then waited for the scheduled handler to finish.

### Why this matters

Agent workflows are concurrent. A completion boundary is safer than sleeping for an arbitrary amount of time.


### Inspect the actual message and runtime

The inspection uses `Spore.to_json()` and Reef statistics rather than a hand-written imitation of their state.


In [ ]:
show_spore(received[0])
show_timeline()
show_json(reef.get_network_stats(), "Reef after delivery")


### What just happened?

The card exposes the message identity, sender, timestamps, and structured knowledge. The timeline shows when the handler started and finished.

### Why this matters

Visible state makes it much easier to debug routing, filtering, and lifecycle problems in larger systems.


In [ ]:
assert received[0].id == spore_id
assert received[0].from_agent == "learner"
assert received[0].knowledge["type"] == "architecture_question"
show_callout("Verified", "The delivered Spore is the one the Reef created.", role="reef")


## Your turn

Add a second observer that listens on the same channel. Broadcast a message with type `practice_message`, wait for completion, and verify that both observers saw it. The scaffold is intentionally commented so certification still runs deterministically.


In [ ]:
# second_received = []
# def second_observer(spore):
#     second_received.append(spore)
# reef.subscribe("second-observer", second_observer, channel=reef.default_channel)
# reef.broadcast("learner", {"type": "practice_message", "value": 1})
# assert reef.wait_for_completion(timeout=10)
# assert len(second_received) == 1


## Common mistake

**Mistake:** Assuming `broadcast()` means every handler has already finished.

**Correction:** Use `wait_for_completion()` before inspecting results or ending the process.


<details>
<summary>Under the hood</summary>

Each Reef channel owns worker infrastructure. `broadcast()` serializes safe message fields, enqueues delivery, and returns an identity. Completion tracking includes work triggered by handlers, which is why it is a stronger boundary than `time.sleep()`.

</details>


## Recap

- Agents specialize; handlers perform their message-driven work.
- Spores carry structured knowledge and identity.
- The Reef routes, tracks completion, and owns channel lifecycle.
- A model runtime is separate and only needed for model-backed work.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
assert reef.shutdown(timeout=10)
stage("reef", "shutdown", "all channel workers closed")
show_timeline()


### Next lesson

Continue to `01_hello_world.ipynb` to turn a decorated Python function into an Agent and build a complete two-agent exchange.
